Task 1

In [ ]:
import pandas as pd

# load data and make a copy
rentals_raw = pd.read_csv('data/bike_rentals.csv')
rentals = rentals_raw.copy()

# check shape and data types
print(rentals.shape)
print(rentals.dtypes)
# price -> str

# find empty columns
empty_cols = rentals.columns[rentals.isna().all()]
print('Empty columns:', empty_cols)

# drop empty columns and check again
rentals = rentals.dropna(axis=1, how='all')
print(rentals.shape)
print(len(rentals.columns[rentals.isna().all()]))

Task 2

In [ ]:
# look at first rows of price
print(rentals['price'].head(10))

# step 1: remove currency text
rentals['price'] = rentals['price'].str.replace('SEK ', '', regex=False)
print(rentals['price'].head(3))

# step 2: remove comma
rentals['price'] = rentals['price'].str.replace(',', '', regex=False)
print(rentals['price'].head(3))

# change type to float and check stats
rentals['price'] = rentals['price'].astype(float)
print(rentals['price'].dtype)
print(rentals['price'].describe())

# show top 3 most expensive rentals
top3 = rentals.sort_values('price', ascending=False).head(3)
print(top3[['rental_id', 'bike_type', 'status', 'price']])

Task 3

In [ ]:
# count missing prices and percentage
missing_count = rentals['price'].isna().sum()
missing_pct = rentals['price'].isna().mean() * 100
print(missing_count, round(missing_pct, 1))

# compare missing prices with cancelled status
print(pd.crosstab(rentals['price'].isna(), rentals['status'] == 'Cancelled'))

# decision: 47 missing prices are cancelled rentals (nothing paid).
# 4 are missing by mistaketats. I will leave them as missing

Task 4

In [ ]:
# find columns with only t and f
tf_cols = [c for c in rentals.columns if set(rentals[c].dropna().unique()) <= {'t', 'f'}]
print(tf_cols)

# why status is not in the list: it has 3 text values (Completed, Cancelled, No-show), not t and f

# map t and f to True and False
for col in tf_cols:
    rentals[col] = rentals[col].map({'t': True, 'f': False})

# check missing values after mapping
print(rentals[tf_cols].isna().sum())

# check member counts and compare median price
print(rentals['is_member'].value_counts())
print(rentals.groupby('is_member')['price'].agg(['count', 'median']))

Task 5

In [ ]:
print(rentals['start_date'].isna().sum())
print(rentals['end_date'].isna().sum())

# convert text to dates
rentals['start_date'] = pd.to_datetime(rentals['start_date'], errors='coerce')
rentals['end_date'] = pd.to_datetime(rentals['end_date'], errors='coerce')

print(rentals['start_date'].isna().sum()) # -> 2
print(rentals['end_date'].isna().sum())
print(rentals[['start_date', 'end_date']].dtypes)

rentals['rental_days'] = (rentals['end_date'] - rentals['start_date']).dt.days
print(rentals['rental_days'].describe())
# min is -2 - two rentals end before they start

Task 6

In [ ]:
# count missing ratings and compare with status
print(rentals['rating'].isna().sum())
print(pd.crosstab(rentals['rating'].isna(), rentals['status']))
# not a perfect match - 46 completed rentals have no rating

# create has_rating column and count values
rentals['has_rating'] = rentals['rating'].notna()
print(rentals['has_rating'].value_counts())

# check stats for rating
print(rentals['rating'].describe())
# median rating is 5.0

Task 7

In [ ]:
# check values in helmets_text
print(rentals['helmets_text'].value_counts(dropna=False))

# create child_helmet column
rentals['child_helmet'] = rentals['helmets_text'].str.contains('child', na=False)

# extract number and replace 'No helmets' with 0
nums = rentals['helmets_text'].str.extract(r'(\d+)', expand=False).astype(float)
rentals['helmets_count'] = nums.where(~rentals['helmets_text'].str.contains('No helmets', na=False), 0)

# "No helmets" has no digits so extract gives NaN; where() sets those rows to 0

# count values in helmets_count and child_helmet
print(rentals['helmets_count'].value_counts(dropna=False))
print(rentals['child_helmet'].value_counts())